# Lab 2c · Investiga la empresa… y lanza tu webapp

Último paso: que el agente decida **qué tienes que preparar** sobre la empresa y lo prepare **con fuentes**. Después
lo juntamos todo en un flujo y lo publicamos en tu webapp.

Tiempo: 15 minutos. ¿Atascado? `mucdia.lab.pista("investigador")` (y `mucdia.lab.solucion("investigador")`).

In [ ]:
import sys

import pandas as pd

import mucdia

sys.path.insert(0, str(mucdia.config.RAIZ / "sesion2"))
import candidatura as C  # noqa: E402

PIEZAS = C.piezas.CARPETA      # tus celdas ✏️ guardan aquí su código: lo usará tu webapp
PIEZAS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 110)
cliente = mucdia.cliente()

oferta, pagina = C.oferta.cargar_oferta(), C.oferta.cargar_pagina()   # las del lab 2a
print(f"Oferta: {oferta.puesto} · {oferta.empresa}")

## Paso 1 · Fuentes que se pueden citar, sin pagar (2')

¿Por qué no «buscar en Google» desde el modelo? La búsqueda integrada de Gemini no entra en el plan gratuito
(responde 429) y Google Noticias prohíbe en su robots.txt que la lean programas. Usamos tres fuentes gratuitas que
lo permiten:

| Herramienta | Fuente | Para qué |
|---|---|---|
| `ficha_empresa` | API de Wikipedia y Wikidata | Qué hace, sector, sede, plantilla, ingresos, web oficial |
| `noticias` | Bing Noticias (RSS) | Titulares del último año, con fecha, medio y enlace |
| `leer_web` | La web (si robots.txt lo permite) | La web oficial, su sala de prensa o una noticia |

In [ ]:
print(C.investigacion.ficha_empresa(oferta.empresa)[:1500], "\n")
print(C.investigacion.noticias(oferta.empresa, maximo=5))

## Paso 2 · ✏️ El puesto de trabajo del investigador (4')

Escribe sus instrucciones. Lo importante: que **decida** qué preparar para ESTA oferta (no lo mismo para todas),
que investigue con un límite de llamadas, que **cite la fuente** de cada dato y que diga lo que no encuentra en vez
de rellenarlo con lo que «suele» pasar.

In [ ]:
INSTRUCCIONES_INVESTIGADOR = """Eres investigador de empresas y preparas a una persona para la entrevista de ESTA oferta.
✏️ TODO:
1. ¿Qué tiene que decidir primero? (qué preparar: temas, prioridad y por qué)
2. ¿Con qué herramientas investiga (ficha_empresa, noticias, leer_web) y con qué límite de llamadas?
3. ¿Cómo escribe el dossier? (¿y las fuentes? ¿y lo que no encuentra?)
4. ¿Qué más prepara? (preguntas que le harán y preguntas para hacerles)"""
C.piezas.guardar_instrucciones("investigador", INSTRUCCIONES_INVESTIGADOR)
mucdia.lab.comprobar_investigador(INSTRUCCIONES_INVESTIGADOR)

## Paso 3 · El investigador decide qué preparar… y lo prepara (3')

In [ ]:
investigador = C.agentes.investigador(cliente)      # con tus instrucciones
mucdia.reiniciar_traza()
r = await investigador.run(f"Hoy es {C.carrera.fecha_referencia():%d-%m-%Y}. Investiga la empresa de esta oferta y "
                           f"prepárame para la entrevista:\n\n{C.agentes.describir_oferta(oferta)}",
                           options={"response_format": C.agentes.Dossier})
dossier = r.value
mucdia.mostrar_traza(r)
con_fuente = sum(1 for s in dossier.secciones if s.fuentes)
print(f"\n📏 {len(dossier.que_preparar)} temas que preparar · {len(dossier.secciones)} secciones investigadas "
      f"({con_fuente} con fuente) · {len(dossier.no_encontrado)} cosas que no ha encontrado")
pd.DataFrame([t.model_dump() for t in dossier.que_preparar])

In [ ]:
for s in dossier.secciones:
    print(f"■ {s.titulo}\n{s.contenido}\n  Fuentes: {', '.join(s.fuentes) or '—'}\n")
print("❓ Te preguntarán:", *dossier.preguntas_probables, sep="\n  - ")
print("🙋 Pregúntales tú:", *dossier.preguntas_para_ellos, sep="\n  - ")
print("🔎 No encontrado:", *dossier.no_encontrado, sep="\n  - ")

## Paso 4 · Todo junto: un flujo con cuatro pasos (2')

Un **flujo** (*workflow*): el código fija los pasos y su orden; en cada paso que necesita juicio trabaja un agente.
El CV y la empresa van **en paralelo**. Es lo que ejecuta tu webapp
([`candidatura/flujo.py`](candidatura/flujo.py)):

```
1 · leer la oferta (código) → estructurarla (analista)
2 · adaptar el CV por MCP (editor)  ‖  investigar la empresa (investigador)
3 · plan de 4 semanas y pitch de 3 frases (coach)
```

In [ ]:
from candidatura import flujo

resultado = await flujo.preparar(pagina=pagina)     # imprime cada paso y cada herramienta
m = resultado["medidas"]
print(f"\n📏 {m['segundos']} s · {m['llamadas']} llamadas al modelo ({m['grabadas']} grabadas) · {C.fmt(m['tokens'])} tokens")
print("\n🎤 Tu pitch:", *resultado["plan"]["pitch"], sep="\n  ")

## Paso 5 · Tu webapp (2')

Abre un **terminal** (menú ☰ → Terminal → New Terminal) y escribe:

```bash
python sesion2/webapp.py
```

En Codespaces, pestaña **Ports**: en la fila «Tu agente de candidaturas» (8000), pulsa el globo 🌐. Pega tu oferta y
mira cómo trabaja el equipo en directo. La tarjeta «Tu perfil» te dice qué piezas son ✏️ tuyas. Puedes subir tu CV
desde ahí. Para pararla: **Ctrl+C** en el terminal.

## Paso 6 · ✏️ ¿Quién escribe mejor tu pitch: tú o tu agente? (2')

Me mandasteis tres frases a mano. Pégalas aquí: un **juez** (otro agente) compara las dos versiones
con una rúbrica, sin saber cuál es cuál. ¿Más rápido es mejor? Mídelo, no lo supongas.

In [ ]:
import random

from agent_framework import Agent

MIS_3_FRASES = """"""   # ✏️ pega aquí las tres frases que me mandaste por email

if MIS_3_FRASES.strip():
    pitch_agente = " ".join(resultado["plan"]["pitch"])
    versiones = [("a mano", MIS_3_FRASES.strip()), ("agente", pitch_agente)]
    random.Random(0).shuffle(versiones)                                  # el juez no sabe cuál es cuál
    juez = Agent(client=cliente, name="juez", instructions=(
        "Eres responsable de selección de perfiles júnior de datos. Comparas dos pitch para una oferta. Puntúa de "
        "1 a 5 cada uno en: pruebas (hechos verificables), encaje (con lo que pide la oferta), empresa (muestra "
        "que la conoce) y honestidad (sin superlativos ni promesas vacías). Di cuál ficharías y por qué, en 3 líneas."))
    r = await juez.run(f"Oferta: {oferta.puesto} en {oferta.empresa}.\n\nPitch 1:\n{versiones[0][1]}\n\n"
                       f"Pitch 2:\n{versiones[1][1]}")
    print(f"(Pitch 1 = {versiones[0][0]} · Pitch 2 = {versiones[1][0]})\n\n{r.text}")

## Para pensar

1. ¿Qué parte de tu candidatura **no** puede hacer el agente? (Pista: lo tácito. Hablar con alguien
   que trabaje allí, por ejemplo.)
2. El pitch del agente, ¿es tuyo? ¿Lo dirías tal cual en una entrevista?
3. Si esta webapp la usaran 1.000 personas, ¿qué podría salir mal? (👉 sesión 3: evaluar, atacar y blindar.)

## ⭐ Para ir más allá

1. **Un verificador de exageraciones**: añade al flujo un agente que compare el CV original con el adaptado y
   marque las frases que dicen más de lo que había. ¿Cuánto cuesta en llamadas? ¿Cuántas encuentra?
2. **Cambia el flujo**: en `candidatura/flujo.py`, haz que el coach reciba también las noticias. ¿Mejora el pitch?
3. Prueba la webapp con otra oferta… y con el CV de una compañera (con su permiso).

In [ ]:
mucdia.uso()